# 消贷客户采样方法选择 模型训练和复用

此 Notebook 使用 Python 3.6 语法。它将采样方法比较、指定方法重新训练和历史模型复用放在同一流程中。

运行模式由 `MODEL_MODE` 控制：

- `select`：在训练集比较采样方法，选择最优方法，保存配置并训练最终模型。
- `refit`：使用指定方法或已有配置重新确定最佳轮数，保存更新后的配置和最终模型。
- `reuse`：加载已保存的配置、预处理器和模型，只在测试集评价。

违约标签为客户最大 `SEQUOVERDAY > 30`。不包含 最终模型、Gain 特征重要性或两份数据重叠分析。


## Cell 1：参数配置（在这里修改路径和参数）

In [ ]:
# ── 文件路径 ──
FILE_PATH   = "consumer_loan0908.csv"   # 第一份数据（Excel 或 CSV）

# ── 快照日期（用于由 brth_dt 计算年龄） ──
SNAPSHOT_DATE = "2026-09-04"

# 已清洗数据开关：False 时从原始提取数据完成清洗；True 时直接读取下方文件。
# 文件应为本 Notebook 第 11 步导出的清洗结果，且保留 cst_id、y_freq、y_dq_risk。
# 支持 .csv、.xlsx、.xls；即使当前只训练一个目标，两个标签也用于保持原有联合分层切分。
SKIP_DATA_CLEANING = False
CLEANED_FILE_PATH = "consumer_loan_cleaned.xlsx"

# ── 建模目标 ──
# 【任务3前置要求】本 Notebook 必须完整运行两次：TARGET 分别设为 y_freq 和 y_dq_risk。
# 两次都要从 Cell 1 运行到最后，并保留 sampling_method_selection_y_freq.csv
# 与 sampling_method_selection_y_dq_risk.csv；任务3需要分别读取两者 selected=True 的结果。
# "y_freq"    : 频率活跃度标签（贷款周期内余额/支用变化）
# "y_dq_risk" : 违约风险标签（客户最大连续拖欠天数 SEQUOVERDAY > 30 → 1）
TARGET = "y_freq"

# ── y_freq 因变量构造模式 ──
# "bout_gt0_and_curr_p80" : ba_out_bal_diff > 0  且  ac_curr_bal_diff >= P80（默认）
# "bout_p80_and_accr_p80" : ba_out_bal_diff >= P80  且  ac_accr_bal_diff >= P80
# "curr_p80_only"         : ac_curr_bal_diff >= P80（单条件，仅看余额差值前20%）
# "curr_p80_and_bout_p80" : ac_curr_bal_diff >= P80  且  ba_out_bal_diff >= P80（两者均前20%）
Y_FREQ_MODE = "curr_p80_and_bout_p80"  # 仅 TARGET="y_freq" 时有效

# ── 授信额度多项式特征 ──
ADD_QUOTA_SQ    = False
ADD_QUOTA_CUBE  = False
ADD_QUOTA_LOG   = False

# ── 到期日筛选开关 ──
APPLY_MATURITY_FILTER = False
MATURITY_CUTOFF       = "2026-09-04"

# ── 贷款生效日筛选：当前开启，仅保留以下日期区间内生效的贷款 ──
APPLY_EFF_DATE_FILTER = False
EFF_DATE_LOWER        = "2025-01-01"
EFF_DATE_UPPER        = "2026-03-31"

# ── LightGBM 建模参数 ──
RANDOM_STATE          = 42
TEST_SIZE             = 0.2   # 旧参数保留兼容；实际按照以下4个比例划分数据集
TRAIN_RATIO          = 0.60
VALIDATION_RATIO     = 0.15
CALIBRATION_RATIO    = 0.15
FINAL_TEST_RATIO     = 0.10
INNER_EARLY_STOP_RATIO = 0.20  # 仅在60%训练集内部用于确定LightGBM迭代轮数
# 候选方法在验证集确定；最终测试按技术文档使用测试集正样本率对应的 Top-ρ 分位数阈值。
# 概率校准集不参与任务2的模型训练或方法选择，留给任务3训练概率校准器。

# ── 贷款账户重复处理 ──
# "none"：只输出重复统计；"exact"：仅删除整行完全相同副本，同键字段冲突时报错；
# "cst_loan_first"：只要 cst_id 和 loanacctno 相同即去重，并按源文件顺序保留第一条。
DEDUP_CST_LOAN_MODE = "none"
# 旧开关保留兼容；True 且 MODE="none" 时等同于 MODE="exact"。
DEDUP_CST_LOAN = False
# 新取数不含起点账户状态；不执行起点违约客户剔除。
EXCLUDE_DQ_START_CUSTOMERS = False
HANDLE_IMBALANCE      = False    # 旧参数保留供原 最终模型 后备训练；候选 baseline 不使用类别权重
EARLY_STOPPING_ROUNDS = 50
LGB_PARAMS = {
    "objective":        "binary",
    "metric":           "auc",
    "n_estimators":     500,
    "learning_rate":    0.05,
    "num_leaves":       31,
    "max_depth":        -1,
    "min_child_samples": 20,
    "subsample":        0.8,
    "bagging_freq":     1,   # subsample 只有在 bagging_freq > 0 时才生效
    "colsample_bytree": 0.8,
    "reg_alpha":        0.1,
    "reg_lambda":       0.1,
    "random_state":     42,
    "verbose":          -1,
}

# ════════════════════════════════════════════════════════
# 采样参数配置
# ════════════════════════════════════════════════════════
# SAMPLING_METHODS：填入想对比的方法列表，留空列表则只跑无采样基线。
# 采样只作用于训练集；验证、校准和测试集始终保持各自时间段的原始分布。
#   过采样: 'smote' / 'borderline_smote' / 'adasyn'
#   欠采样: 'random_under' / 'tomek' / 'enn'
#   组合:   'smoteenn' / 'smotetomek'
#   集成:   'balance_cascade'（分类器驱动级联）/ 'easy_ensemble'（独立随机欠采样集成）
SAMPLING_METHODS = ["random_over", "smote", "borderline_smote",'adasyn','smoteenn','smotetomek', "random_under", "balance_cascade", "easy_ensemble"]  # ← 手动控制

# 普通采样方法目标 正样本数/负样本数；1.0 = 训练集采样后正负 1:1
SAMPLING_STRATEGY = 1.0
# BalanceCascade / Ensemble 子模型数量，以及每个子集中 负样本数/正样本数；1.0 = 每个子集正负 1:1
SAMPLING_N_ESTIMATORS = 10
SAMPLING_ENSEMBLE_RATIO = 1.0

# 名义类别字段的显式候选；age 是连续变量，不得放入该列表。
# SMOTE/SMOTEENN/SMOTETomek 按 SMOTENC 规则处理；BorderlineSMOTE/ADASYN 是混合特征扩展。
SAMPLING_CATEGORICAL_CANDIDATES = [
    "gnd_cd", "mar_sttn_cd", "eddgr_cd", "ocp_cd", "cst_star_cd",
    "busikind",
]

# ════════════════════════════════════════════════════════
# 训练、选型与复用模式（Python 3.6）
# ════════════════════════════════════════════════════════
# select：比较 SAMPLING_METHODS，保存最优采样方法、参数和轮数，并训练最终模型。
# refit ：只训练一个方法并更新最佳轮数；FORCE_METHOD=True 时使用下方指定方法。
# reuse ：复用已保存的配置、预处理器和模型；不重新训练。
MODEL_MODE = "select"
if MODEL_MODE not in ("select", "refit", "reuse"):
    raise ValueError("MODEL_MODE 仅支持 select / refit / reuse")
RUN_MODEL_SELECTION = MODEL_MODE == "select"
REFIT_BEST_ROUNDS_ONLY = MODEL_MODE == "refit"

# 仅在 MODEL_MODE="refit" 且 FORCE_METHOD=True 时生效。
FORCE_METHOD = False
FORCED_METHOD_KEY = "easy_ensemble"
FORCED_METHOD_LABEL = "Easy Ensemble（随机欠采样集成）"
# refit 且 FORCE_METHOD=False 时：True 使用本单元的 LGB_PARAMS；False 沿用历史参数。
USE_CURRENT_LGB_PARAMS = False
# reuse 模式无历史配置时的兜底轮数；通常保持 None。
FORCED_ROUNDS = None

# 训练完成后保存：采样配置、LightGBM 模型、训练集拟合的预处理器和元数据。
SELECTED_CONFIG_PATH = "selected_model_config_{0}.json".format(TARGET)
FULL_MODEL_ACTION = "auto"  # auto：有匹配制品则加载；train：重训覆盖；load：只加载
ARTIFACT_ROOT = "full_model_{0}".format(TARGET)
MODEL_DIR = ARTIFACT_ROOT
PREPROCESSOR_PATH = "model_preprocessor_{0}.pkl".format(TARGET)
METADATA_PATH = "full_model_metadata_{0}.json".format(TARGET)


## Cell 2：导入模块

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 中文字体（离线机器可换为 'SimHei' 或 'Source Han Sans CN'）
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 预处理模块（与本 notebook 放在同一目录）
from load_consumer_loan_data import (
    load_kechuang_potential, get_feature_stats,
    get_feature_descriptive_stats, get_consumption_field_cn_map,
    PotentialFeaturePreprocessor, stratified_dual_target_partition_indices,
)

print("模块导入完成")

## Cell 3：运行预处理

In [ ]:
# 数据读取：可由开关直接使用此前导出的清洗结果
import os

if SKIP_DATA_CLEANING:
    if not os.path.exists(CLEANED_FILE_PATH):
        raise FileNotFoundError("未找到已清洗数据文件：{0}".format(CLEANED_FILE_PATH))

    file_ext = os.path.splitext(CLEANED_FILE_PATH)[1].lower()
    if file_ext == ".csv":
        df_clean = pd.read_csv(CLEANED_FILE_PATH)
    elif file_ext in (".xlsx", ".xls"):
        df_clean = pd.read_excel(CLEANED_FILE_PATH)
    else:
        raise ValueError("CLEANED_FILE_PATH 仅支持 .csv、.xlsx 或 .xls 文件")

    # Excel/CSV 读取后统一字段名，避免大小写或空格造成后续字段找不到。
    df_clean.columns = [str(col).strip().lower() for col in df_clean.columns]
    required_columns = {"cst_id", "y_freq", "y_dq_risk", TARGET}
    missing_columns = sorted(required_columns.difference(df_clean.columns))
    if missing_columns:
        raise KeyError(
            "已清洗数据缺少字段：{0}。请使用本 Notebook 导出的清洗文件。".format(
                ", ".join(missing_columns)
            )
        )

    for label_col in ("y_freq", "y_dq_risk"):
        df_clean[label_col] = pd.to_numeric(df_clean[label_col], errors="coerce")
        invalid = ~df_clean[label_col].isin([0, 1])
        if invalid.any():
            raise ValueError("已清洗数据的 {0} 必须为 0/1，发现 {1} 条无效记录。".format(label_col, int(invalid.sum())))
        df_clean[label_col] = df_clean[label_col].astype(int)

    y = df_clean[TARGET].copy()
    excluded = {"cst_id", "y_freq", "y_dq_risk", "split_eff_date"}
    feature_names = [col for col in df_clean.columns if col not in excluded]
    X = df_clean[feature_names].copy()
    feature_missing_df = get_feature_stats(df_clean, target=TARGET)
    label_stats_df = df_clean[["y_freq", "y_dq_risk"]].agg(["count", "sum", "mean"]).T.reset_index()
    label_stats_df.columns = ["label", "count", "positive_count", "positive_rate"]
    thresholds = {"source": "precleaned_file", "target": TARGET}
    field_coverage_summary_df = pd.DataFrame()
    field_coverage_monthly_df = pd.DataFrame()
    print("已跳过数据清洗，直接读取：{0}".format(CLEANED_FILE_PATH))
else:
    (
        df_clean,
        feature_missing_df,
        label_stats_df,
        X,
        y,
        feature_names,
        thresholds,
        field_coverage_summary_df,
        field_coverage_monthly_df,
    ) = load_kechuang_potential(
        FILE_PATH,
        snapshot_date=SNAPSHOT_DATE,
        target=TARGET,
        y_freq_mode=Y_FREQ_MODE,
        y_freq_threshold=Y_FREQ_THRESHOLD,
        y_freq_cumulative_rate=Y_FREQ_CUMULATIVE_RATE,
        feature_period_mode=FEATURE_PERIOD_MODE,
        customer_aggregation=CUSTOMER_AGGREGATION,
        keep_only_active_at_snapshot=KEEP_ONLY_ACTIVE_AT_SNAPSHOT,
        exclude_dq_start_customers=EXCLUDE_DQ_START_CUSTOMERS,
        required_feature_columns=REQUIRED_FEATURE_COLUMNS,
        optional_feature_columns=OPTIONAL_FEATURE_COLUMNS,
        require_dual_label_cohort=REQUIRE_DUAL_LABEL_COHORT,
        dedup_cst_loan_mode=DEDUP_CST_LOAN_MODE,
        dedup_cst_loan=DEDUP_CST_LOAN,
        drop_feature_columns=DROP_FEATURE_COLUMNS,
    )


## 字段覆盖与全部特征描述统计


In [ ]:
# 字段日期覆盖统计在读取后完成列名小写/来源特征字段改名时立即计算，
# 尚未经过贷款筛选、去重或一客多贷聚合。
print('字段覆盖汇总（稳定覆盖起始月：连续3个月覆盖率均达到90%的最早月份）')
display(field_coverage_summary_df)

print('字段按贷款生效月份覆盖明细')
display(field_coverage_monthly_df)

# 清洗结束后的全部原始特征，不包含因变量、客户ID和标签阈值参考期辅助列。
_desc_exclude = {'cst_id', 'y_freq', 'y_dq_risk', 'split_eff_date'}
_desc_feature_cols = [c for c in df_clean.columns if c not in _desc_exclude]
numeric_feature_desc_df, categorical_feature_desc_df = get_feature_descriptive_stats(
    df_clean, _desc_feature_cols
)

print('数值型特征描述统计（最小值、最大值、平均值）')
display(numeric_feature_desc_df)

print('类别型特征描述统计（众数及每个类别比例）')
display(categorical_feature_desc_df)

# ── 保存完整统计结果：Notebook页面可能截断，CSV文件保留全部行 ──
from pathlib import Path
STATS_OUTPUT_DIR = Path(f'字段覆盖与全部特征描述性统计结果_{TARGET}')
STATS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_stats_tables = {
    '字段覆盖汇总.csv': field_coverage_summary_df,
    '字段月度覆盖明细.csv': field_coverage_monthly_df,
    '数值型特征描述统计.csv': numeric_feature_desc_df,
    '类别型特征描述统计.csv': categorical_feature_desc_df,
}
for _file_name, _table in _stats_tables.items():
    _table.to_csv(STATS_OUTPUT_DIR / _file_name, index=False, encoding='utf-8-sig')

print(f'\n✅ 完整统计结果已保存至文件夹: {STATS_OUTPUT_DIR.resolve()}')
print(f'  字段覆盖汇总      : {len(field_coverage_summary_df):,} 行')
print(f'  字段月度覆盖明细  : {len(field_coverage_monthly_df):,} 行')
print(f'  数值型特征描述统计: {len(numeric_feature_desc_df):,} 行')
print(f'  类别型特征描述统计: {len(categorical_feature_desc_df):,} 行')


## Cell 4：数据规模概览

In [ ]:
print("━" * 50)
print(f"  客户数（行数）     : {len(df_clean):,}")
print(f"  特征数（建模用）    : {len(feature_names):,}")
print(f"  清洗后总列数        : {df_clean.shape[1]:,}")
print("━" * 50)
print(f"  建模目标            : {TARGET}")
print(f"  到期日筛选          : {'开启，cutoff=' + MATURITY_CUTOFF if APPLY_MATURITY_FILTER else '关闭'}")
if TARGET == 'y_freq':
    print(f"  y_freq 构造模式     : {thresholds['y_freq_mode']}")
    if thresholds['y_freq_mode'] == 'bout_gt0_and_curr_p80':
        print(f"  y_freq 条件①       : ba_out_bal_diff > 0")
        print(f"  y_freq 条件②       : ac_curr_bal_diff >= {thresholds['thr_curr']:.4f}（P80）")
    elif thresholds['y_freq_mode'] == 'bout_p80_and_accr_p80':
        print(f"  y_freq 条件①       : ba_out_bal_diff  >= {thresholds['thr_bout']:.4f}（P80）")
        print(f"  y_freq 条件②       : ac_accr_bal_diff >= {thresholds['thr_accr']:.4f}（P80）")
    elif thresholds['y_freq_mode'] == 'curr_p80_only':
        print(f"  y_freq 条件         : ac_curr_bal_diff >= {thresholds['thr_curr']:.4f}（P80，单条件）")
    else:  # curr_p80_and_bout_p80
        print(f"  y_freq 条件①       : ac_curr_bal_diff >= {thresholds['thr_curr']:.4f}（P80）")
        print(f"  y_freq 条件②       : ba_out_bal_diff  >= {thresholds['thr_bout']:.4f}（P80）")
    print(f"  正样本率            : {y.mean():.4%}  ({y.sum()} / {len(y)})")
else:  # y_dq_risk
    print(f"  y_dq_risk 构造规则  : SEQUOVERDAY > 30 → 1")
    print(f"  正样本率            : {y.mean():.4%}  ({y.sum()} / {len(y)})")
print("━" * 50)


## Cell 5：因变量统计

In [ ]:
print(f"因变量统计（{TARGET}）：")
display(label_stats_df)


## Cell 6：特征缺失率总览

In [ ]:
print(f"共 {len(feature_missing_df)} 个特征")
print("\n── 缺失率 > 0 的特征 ──")
nonzero_miss = feature_missing_df[feature_missing_df['missing_rate'] > 0]
if len(nonzero_miss) == 0:
    print("  所有特征缺失率均为 0（已填充完毕）")
else:
    display(nonzero_miss.reset_index(drop=True))

print("\n── 缺失率为 0 的特征数 ──")
print(f"  {(feature_missing_df['missing_rate'] == 0).sum()} 个")

## Cell 7：特征缺失率可视化（Top 30）

In [ ]:
top_miss = feature_missing_df[feature_missing_df['missing_rate'] > 0].head(30)

if len(top_miss) == 0:
    print("所有特征缺失率均为 0，无需绘图")
else:
    fig, ax = plt.subplots(figsize=(10, max(4, len(top_miss) * 0.35)))
    bars = ax.barh(
        top_miss['feature'][::-1],
        top_miss['missing_rate'][::-1] * 100,
        color='steelblue', edgecolor='white'
    )
    ax.set_xlabel('缺失率 (%)')
    ax.set_title('特征缺失率 Top 30')
    ax.axvline(x=40, color='red', linestyle='--', linewidth=1, label='40% 参考线')
    ax.legend()
    for bar, val in zip(bars, top_miss['missing_rate'][::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val*100:.1f}%', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()

## Cell 8：完整特征列表（全部打印）

In [ ]:
print(f"建模特征共 {len(feature_names)} 个：\n")
for i, name in enumerate(feature_names, 1):
    miss_row = feature_missing_df[feature_missing_df['feature'] == name]
    miss_str = miss_row['missing_rate_pct'].values[0] if len(miss_row) > 0 else "0.00%"
    print(f"  {i:3d}. {name:<40s}  缺失率: {miss_str}")

## Cell 9：清洗后数据预览

In [ ]:
display(df_clean.head(5))

## Cell 10：ba_out_bal_diff 数值分布统计

In [ ]:
# ── ba_out_bal_diff 数值分布统计 ──
# 依赖：Cell 3 已运行，df_clean 中存在 ba_out_bal_diff 列（因变量构造前保留，Part2 清洗后已删除）
# 注意：若 df_clean 中已无此列，请改用 Cell 3 运行前的中间结果，或在 build_labels_potential 输出中查看打印统计

import numpy as np

_col = "ba_out_bal_diff"

if _col not in df_clean.columns:
    print(f"⚠️  df_clean 中不存在 {_col}（已在 Part2 清洗中删除）")
    print("   请参考 Cell 3 运行输出中的'关键字段分布统计'部分查看分位数信息")
else:
    _s = df_clean[_col].dropna()
    _n = len(df_clean)

    print("=" * 52)
    print(f"【{_col} 基本描述统计】")
    print(df_clean[_col].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95, .99]))

    print("\n" + "=" * 52)
    print("【特殊值计数】")
    _null = df_clean[_col].isnull().sum()
    _zero = (_s == 0).sum()
    _pos  = (_s >  0).sum()
    _neg  = (_s <  0).sum()
    print(f"  总行数        : {_n:,}")
    print(f"  缺失值 (NaN)  : {_null:,}  ({_null/_n:.2%})")
    print(f"  等于 0        : {_zero:,}  ({_zero/_n:.2%})")
    print(f"  大于 0 (支用) : {_pos:,}  ({_pos/_n:.2%})")
    print(f"  小于 0 (还款) : {_neg:,}  ({_neg/_n:.2%})")

    print("\n" + "=" * 52)
    print("【分桶分布】")
    _raw_bins = [_s.min(), -10000, -1000, -100, 0, 100, 1000, 10000, _s.max()]
    _bins = sorted(set(_raw_bins))
    try:
        _cut = pd.cut(_s, bins=_bins, include_lowest=True, right=True)
        _dist = _cut.value_counts(sort=False).reset_index()
        _dist.columns = ["区间", "计数"]
        _dist["占比"] = (_dist["计数"] / _n * 100).round(2).astype(str) + "%"
        print(_dist.to_string(index=False))
    except Exception as _e:
        print(f"  分桶失败（数据范围可能过窄）: {_e}")

    print("\n" + "=" * 52)
    print("【非零样本分位数】")
    _nz = _s[_s != 0]
    if len(_nz) > 0:
        for _p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
            print(f"  P{_p:>2}  : {np.percentile(_nz, _p):>15,.2f}")
    else:
        print("  无非零样本")

## Cell 11：保存清洗后数据

In [ ]:
# 保存完整清洗数据（含 cst_id、y 列、所有特征）
OUTPUT_PATH = CLEANED_FILE_PATH
df_clean.to_excel(OUTPUT_PATH, index=False) if not SKIP_DATA_CLEANING else None
print(f"已保存清洗后数据至: {OUTPUT_PATH}")
print(f"  行数: {len(df_clean):,}，列数: {df_clean.shape[1]:,}")


## Cell 12：划分 + 导入

In [ ]:
# ── Cell 12：双标签联合分层随机划分 + 训练集内部早停划分 + 导入 ──

import lightgbm as lgb
import numpy as np
import pandas as pd
import json
import os
import joblib
from sklearn.metrics import (
    roc_auc_score, accuracy_score, recall_score, precision_score,
    f1_score, confusion_matrix, roc_curve,
    precision_recall_curve, average_precision_score, matthews_corrcoef
)
from sampling_methods import BalanceCascade, sampler_factory, print_sampling_summary
# RobustScaler 只拟合采样器收到的训练数据，仅用于近邻距离；原始特征尺度仍供 LightGBM 使用。

OVERALL_POS_RATE = float(y.mean())
if not np.isclose(
    TRAIN_RATIO + VALIDATION_RATIO + CALIBRATION_RATIO + FINAL_TEST_RATIO, 1.0
):
    raise ValueError("四段切分比例之和必须等于 1。")

if not (len(df_clean) == len(X) == len(y)):
    raise AssertionError("df_clean、X、y 行数不一致，无法按同一位置切分。")

# 与额度优化任务一致：按 y_freq × y_dq_risk 联合标签分层随机切成 60% / 15% / 15% / 10%。
# 因为两个 TARGET 共用同一次联合分层，分别运行时会得到完全相同的客户划分。
# y_freq 的数据驱动阈值仍在加载阶段仅用最早 TRAIN_RATIO 客户拟合；它不是模型训练集。
train_pos, val_pos, cal_pos, test_pos = stratified_dual_target_partition_indices(
    df_clean,
    (TRAIN_RATIO, VALIDATION_RATIO, CALIBRATION_RATIO, FINAL_TEST_RATIO),
    random_state=RANDOM_STATE,
)
_row_index = df_clean.index.to_numpy()
train_rows = pd.Series(_row_index[train_pos], name='row_id')
val_rows = pd.Series(_row_index[val_pos], name='row_id')
cal_rows = pd.Series(_row_index[cal_pos], name='row_id')
test_rows = pd.Series(_row_index[test_pos], name='row_id')
y_train = y.loc[train_rows.to_numpy()].copy()
y_val = y.loc[val_rows.to_numpy()].copy()
y_cal = y.loc[cal_rows.to_numpy()].copy()
y_test = y.loc[test_rows.to_numpy()].copy()

# 快速路径优先复用与全量模型配套保存的预处理器；首次运行或强制训练时重新拟合。
_model_files_exist = os.path.isdir(MODEL_DIR) and any(
    name.startswith("model_") and name.endswith(".txt") for name in os.listdir(MODEL_DIR)
)
_artifacts_ready = (
    os.path.isfile(PREPROCESSOR_PATH)
    and os.path.isfile(METADATA_PATH)
    and _model_files_exist
)
if RUN_MODEL_SELECTION and REFIT_BEST_ROUNDS_ONLY:
    raise ValueError("RUN_MODEL_SELECTION 与 REFIT_BEST_ROUNDS_ONLY 不能同时为 True。")
if FULL_MODEL_ACTION not in {"auto", "train", "load"}:
    raise ValueError("FULL_MODEL_ACTION 只能是 'auto'、'train' 或 'load'。")
if FULL_MODEL_ACTION == "load" and (RUN_MODEL_SELECTION or REFIT_BEST_ROUNDS_ONLY):
    raise ValueError("更新方法/轮数后必须重训全量模型；请将 FULL_MODEL_ACTION 改为 'auto' 或 'train'。")
if FULL_MODEL_ACTION == "load" and not _artifacts_ready:
    raise FileNotFoundError("load 模式要求模型目录、预处理器和元数据全部存在。")
_configuration_updated = RUN_MODEL_SELECTION or REFIT_BEST_ROUNDS_ONLY
_load_saved_artifacts = FULL_MODEL_ACTION == "load" or (
    FULL_MODEL_ACTION == "auto" and _artifacts_ready and not _configuration_updated
)

if _load_saved_artifacts:
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        _saved_metadata = json.load(f)
    if _saved_metadata.get("target") != TARGET:
        raise ValueError("保存制品的 TARGET 与当前 TARGET 不一致。")
    model_preprocessor = joblib.load(PREPROCESSOR_PATH)
    print(f"✅ 已加载预处理器：{PREPROCESSOR_PATH}")
else:
    model_preprocessor = PotentialFeaturePreprocessor(
        target=TARGET,
        add_quota_sq=ADD_QUOTA_SQ,
        add_quota_cube=ADD_QUOTA_CUBE,
        add_quota_log=ADD_QUOTA_LOG,
        categorical_features=SAMPLING_CATEGORICAL_CANDIDATES,
    ).fit(df_clean.loc[train_rows.to_numpy()])
    print("✅ 已在训练集拟合新的预处理器")
X_train = model_preprocessor.transform(df_clean.loc[train_rows.to_numpy()])
X_val = model_preprocessor.transform(df_clean.loc[val_rows.to_numpy()])
X_cal = model_preprocessor.transform(df_clean.loc[cal_rows.to_numpy()])
X_test = model_preprocessor.transform(df_clean.loc[test_rows.to_numpy()])
X = model_preprocessor.transform(df_clean)
feature_names = list(model_preprocessor.feature_names_)
label_encoders = model_preprocessor.label_encoders_

# 只在60%训练集内部再划分实际拟合子集和早停集。
# 15%模型选择验证集不再参与LightGBM早停，只用于候选采样方法比较。
if not 0 < INNER_EARLY_STOP_RATIO < 1:
    raise ValueError("INNER_EARLY_STOP_RATIO 必须在 (0, 1) 内。")
inner_train_pos, early_stop_pos = stratified_dual_target_partition_indices(
    df_clean.loc[train_rows.to_numpy()],
    (1.0 - INNER_EARLY_STOP_RATIO, INNER_EARLY_STOP_RATIO),
    random_state=RANDOM_STATE + 1,
)
train_row_array = train_rows.to_numpy()
inner_train_rows = train_row_array[inner_train_pos]
early_stop_rows = train_row_array[early_stop_pos]
X_inner_train = X_train.loc[inner_train_rows]
y_inner_train = y_train.loc[inner_train_rows]
X_early_stop = X_train.loc[early_stop_rows]
y_early_stop = y_train.loc[early_stop_rows]

# 配置项之外，自动纳入由训练集预处理器识别出的名义类别列；档位保持有序数值。
_categorical_candidates = set(SAMPLING_CATEGORICAL_CANDIDATES)
SAMPLING_CATEGORICAL_FEATURES = [
    c for c in feature_names
    if c in _categorical_candidates or c in model_preprocessor.categorical_features_
]
print("预处理器高缺失删列:", model_preprocessor.dropped_high_missing_)
print("采样/LightGBM 按类别处理的字段:", SAMPLING_CATEGORICAL_FEATURES)

# 保留旧变量名，避免后续原有分析单元失效。
X_train_half, X_test_half = X_train, X_test
y_train_half, y_test_half = y_train, y_test

split_rows = []
for split_name, y_part in [
    ("训练集", y_train),
    ("模型选择验证集", y_val),
    ("概率校准集", y_cal),
    ("最终测试集", y_test),
]:
    split_rows.append({
        "集合": split_name,
        "客户数": len(y_part),
        "正样本数": int(y_part.sum()),
        "正样本率": float(y_part.mean()),
    })
split_summary_df = pd.DataFrame(split_rows)
display(split_summary_df)

inner_split_summary_df = pd.DataFrame([
    {"集合": "训练集内部拟合子集", "客户数": len(y_inner_train),
     "正样本数": int(y_inner_train.sum()), "正样本率": float(y_inner_train.mean())},
    {"集合": "训练集内部早停集", "客户数": len(y_early_stop),
     "正样本数": int(y_early_stop.sum()), "正样本率": float(y_early_stop.mean())},
])
display(inner_split_summary_df)

for split_name, y_part in [("训练集", y_train), ("模型选择验证集", y_val), ("概率校准集", y_cal), ("最终测试集", y_test)]:
    if y_part.nunique() < 2:
        raise ValueError(f"{split_name} 只含一个类别，无法训练或计算 AUC。")

for split_name, y_part in [("训练集内部拟合子集", y_inner_train), ("训练集内部早停集", y_early_stop)]:
    if y_part.nunique() < 2:
        raise ValueError(f"{split_name} 只含一个类别，无法执行内部早停。")

print(f"整体样本: {len(y):,}  正样本率={OVERALL_POS_RATE:.4%}  ({int(y.sum())} 正)")
print("四个集合按双标签联合分层随机划分；两个 TARGET 共用客户划分；采样只作用于模型训练数据。")
print("LightGBM早停只使用60%训练集的内部早停子集；模型选择验证集不参与早停。")
print("概率校准集不参与任务2模型训练或方法选择；最终测试按测试集正样本率对应分位数确定Top-ρ阈值。")


def _safe_precision_recall(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    rec = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    pre = float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0
    return pre, rec


def _predict_model(model, X_eval):
    X_arr = X_eval.values if hasattr(X_eval, "values") else np.asarray(X_eval)
    if isinstance(model, (list, tuple)):
        probabilities = [m.predict(X_arr) for m in model]
        return np.mean(np.vstack(probabilities), axis=0)
    return model.predict(X_arr)


def evaluate_model(
    model, X_eval, y_eval, label, split_name="评估集",
    fixed_threshold=None, target_positive_rate=None, threshold_source=None,
):
    """计算排序与分类指标；最终测试阈值按测试集正样本率对应的预测概率分位数确定。"""
    prob = _predict_model(model, X_eval)
    y_arr = y_eval.values if hasattr(y_eval, "values") else np.asarray(y_eval)
    actual_rate = float(np.mean(y_arr))
    if fixed_threshold is None:
        target_positive_rate = (
            actual_rate if target_positive_rate is None else float(target_positive_rate)
        )
        threshold = float(np.quantile(prob, 1 - target_positive_rate))
        threshold_source = threshold_source or split_name
    else:
        threshold = float(fixed_threshold)
        target_positive_rate = (
            float(target_positive_rate) if target_positive_rate is not None else np.nan
        )
        threshold_source = threshold_source or "外部固定阈值"
    pred = (prob >= threshold).astype(int)
    predicted_positive_rate = float(np.mean(pred))

    precision, recall = _safe_precision_recall(y_arr, pred)
    # 兼容旧版scikit-learn：不向f1_score传入zero_division，直接用已安全计算的P/R求F1
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    beta = 2
    f2 = ((1 + beta**2) * precision * recall / (beta**2 * precision + recall)
          if (beta**2 * precision + recall) > 0 else 0.0)
    mcc = matthews_corrcoef(y_arr, pred)
    cm = confusion_matrix(y_arr, pred, labels=[0, 1])
    acc = accuracy_score(y_arr, pred)
    auc = roc_auc_score(y_arr, prob)
    auprc = average_precision_score(y_arr, prob)
    print(f"\n{'='*70}")
    print(f"  {split_name}评估：{label}")
    print(f"{'='*70}")
    print(f"  实际正样本率: {actual_rate:.4%}")
    print(f"  AUC={auc:.4f}  AUPRC={auprc:.4f}")
    print(f"  阈值={threshold:.6f}（来源: {threshold_source}）")
    print(f"  预测正样本率: {predicted_positive_rate:.4%}")
    print(f"  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}")
    print(f"  F2={f2:.4f}  MCC={mcc:.4f}  Accuracy={acc:.4f}")
    print(f"  混淆矩阵: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")

    return dict(
        label=label, split=split_name, auc=auc, auprc=auprc, baseline=actual_rate,
        recall=recall, precision=precision, f1=f1, f2=f2, mcc=mcc, acc=acc,
        threshold=threshold, threshold_positive_rate=target_positive_rate,
        threshold_source=threshold_source, predicted_positive_rate=predicted_positive_rate,
        tn=int(cm[0,0]), fp=int(cm[0,1]), fn=int(cm[1,0]), tp=int(cm[1,1]),
        prob=prob, pred=pred, y_te=y_arr,
    )


def plot_cumulative_gain(prob, y_true, title="累积增益图", save_path="cumulative_gain.png",
                         table_save_path="cumulative_gain_table.csv"):
    """按指定 Top K% 输出累积增益表，并以这些业务节点绘制平滑累积增益曲线。"""
    y_arr = y_true.values if hasattr(y_true, "values") else np.asarray(y_true)
    prob_arr = np.asarray(prob, dtype=float)
    if len(y_arr) != len(prob_arr):
        raise ValueError("累积增益图的预测概率与真实标签长度不一致。")
    if len(y_arr) == 0:
        raise ValueError("测试集为空，无法绘制累积增益图。")

    sort_idx = np.argsort(prob_arr)[::-1]
    y_sorted = y_arr[sort_idx]
    n = len(y_sorted)
    total_pos = int(y_sorted.sum())
    top_k_list = [1, 3, 5, 10, 20, 30, 50]

    rows = []
    for k in top_k_list:
        top_n = min(n, max(1, int(np.ceil(n * k / 100))))
        top_pos = int(y_sorted[:top_n].sum())
        recall_pct = top_pos / total_pos * 100 if total_pos > 0 else 0.0
        top_positive_rate = top_pos / top_n * 100
        rows.append({
            "Top K%": f"Top {k}%",
            "召回正样本（%）": recall_pct,
            "Top K%客户中真实正样本比例（%）": top_positive_rate,
        })

    gain_table = pd.DataFrame(rows)
    from IPython.display import display
    print("\n累积增益表（测试集）：")
    display(gain_table.style.format({
        "召回正样本（%）": "{:.2f}",
        "Top K%客户中真实正样本比例（%）": "{:.2f}",
    }))
    gain_table.to_csv(table_save_path, index=False, encoding="utf-8-sig")
    print(f"累积增益表已保存: {table_save_path}")

    # 以业务指定的 Top K% 节点绘制保形平滑曲线，避免逐客户连接造成折线感。
    x_nodes = np.array([0] + top_k_list + [100], dtype=float)
    y_nodes = np.array([0] + gain_table["召回正样本（%）"].tolist() + [100.0 if total_pos > 0 else 0.0])
    x_smooth = np.linspace(0, 100, 500)
    try:
        from scipy.interpolate import PchipInterpolator
        y_smooth = PchipInterpolator(x_nodes, y_nodes)(x_smooth)
    except ImportError:
        # 兼容未安装 scipy 的环境；仍只基于指定节点绘图。
        y_smooth = np.interp(x_smooth, x_nodes, y_nodes)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(x_smooth, y_smooth, color="steelblue", lw=2.5, label="模型累积增益曲线（指定Top K%节点平滑）")
    ax.scatter(x_nodes[1:-1], y_nodes[1:-1], color="steelblue", s=45, zorder=4, label="Top K%业务节点")
    ax.plot([0, 100], [0, 100], color="gray", lw=1.5, linestyle="--", label="随机基线（无模型）")
    for x, y_value in zip(x_nodes[1:-1], y_nodes[1:-1]):
        ax.annotate(f"Top {int(x)}%\n召回 {y_value:.1f}%", (x, y_value),
                    xytext=(4, -14), textcoords="offset points", fontsize=8, color="steelblue")
    ax.set_xlabel("覆盖客户比例（按预测概率从高到低，%）")
    ax.set_ylabel("召回正样本比例（%）")
    ax.set_title(title)
    ax.legend(loc="lower right", fontsize=9)
    ax.set_xlim([0, 100]); ax.set_ylim([0, 105]); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"图片已保存: {save_path}")
    return gain_table

print("\n✅ 双标签联合分层随机切分完成；evaluate_model / plot_cumulative_gain 已定义")


## Cell 13A：采样与 LightGBM 训练函数定义（快速路径必须运行）


In [ ]:
# ── Cell 13A：仅定义函数，不执行候选方法训练 ──
def _normalise_method_name(method):
    if method is None:
        return "baseline", "无采样基线"
    method_key = str(method).lower().replace("-", "_").replace(" ", "_")
    if method_key in ("ensemble", "easyensemble"):
        method_key = "easy_ensemble"
    elif method_key == "balancecascade":
        method_key = "balance_cascade"
    labels = {
        "balance_cascade": "Balance Cascade（分类器驱动）",
        "easy_ensemble": "Easy Ensemble（随机欠采样集成）",
        "smoteenn": "类别安全 SMOTE + ENN",
        "smotetomek": "类别安全 SMOTE + Tomek",
    }
    return method_key, labels.get(method_key, method_key)


def _sample_training_data(method_key, X_base, y_base):
    if method_key == "baseline":
        return (X_base, y_base)
    if method_key in ("balance_cascade", "easy_ensemble"):
        sampler = sampler_factory(
            method=method_key,
            random_state=RANDOM_STATE,
            n_estimators=SAMPLING_N_ESTIMATORS,
            ratio=SAMPLING_ENSEMBLE_RATIO,
            categorical_features=SAMPLING_CATEGORICAL_FEATURES,
        )
    else:
        sampler = sampler_factory(
            method=method_key,
            sampling_strategy=SAMPLING_STRATEGY,
            random_state=RANDOM_STATE,
            categorical_features=SAMPLING_CATEGORICAL_FEATURES,
        )
    if method_key == "balance_cascade":
        # 子集必须由随后真正训练并集成的 LightGBM 逐轮驱动，不能提前用代理模型生成。
        return sampler.initialize(X_base, y_base)
    return sampler.fit_resample(X_base, y_base)

def _train_one_lgb(Xtr_use, ytr_use, X_valid, y_valid, label, params_base, num_rounds=None):
    params_local = params_base.copy()
    params_local.pop("scale_pos_weight", None)
    params_local.pop("early_stopping_rounds", None)
    configured_rounds = params_local.pop("n_estimators", LGB_PARAMS["n_estimators"])
    rounds = int(configured_rounds if num_rounds is None else num_rounds)
    training_columns = list(Xtr_use.columns) if hasattr(Xtr_use, "columns") else feature_names
    categorical_features_lgb = [
        c for c in SAMPLING_CATEGORICAL_FEATURES if c in training_columns
    ]
    dtrain = lgb.Dataset(
        Xtr_use, label=ytr_use, feature_name=feature_names,
        categorical_feature=categorical_features_lgb,
    )

    if X_valid is not None:
        dvalid = lgb.Dataset(X_valid, label=y_valid, feature_name=feature_names, reference=dtrain)
        booster_one = lgb.train(
            params=params_local,
            train_set=dtrain,
            num_boost_round=rounds,
            valid_sets=[dtrain, dvalid],
            valid_names=["train", "validation"],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=100,
        )
    else:
        booster_one = lgb.train(
            params=params_local,
            train_set=dtrain,
            num_boost_round=rounds,
            verbose_eval=False,
        )
    print(f"  {label} 迭代轮次: {booster_one.best_iteration or rounds}")
    return booster_one


def _train_sampled(sampled, X_valid, y_valid, scene_label, params_base, round_limits=None):
    def _rounds_for(model_index):
        if isinstance(round_limits, (list, tuple, np.ndarray)):
            if len(round_limits) == 0:
                return None
            # 开发集的有效级联轮数可能与训练集不同；超出时复用最后一个早停轮数。
            return round_limits[min(model_index, len(round_limits) - 1)]
        return round_limits

    if isinstance(sampled, BalanceCascade):
        models = []
        while sampled.has_next_subset():
            model_index = len(models)
            X_sub, y_sub = sampled.next_subset()
            rounds = _rounds_for(model_index)
            model = _train_one_lgb(
                X_sub, y_sub, X_valid, y_valid,
                f"{scene_label} | model {model_index+1}/{sampled.effective_n_estimators_}",
                params_base, rounds,
            )
            models.append(model)
            # 用本轮实际 LightGBM 在当前多数类池上的分数调整阈值并删除易分样本。
            remaining_prob = model.predict(sampled.remaining_X())
            cascade_info = sampled.update(remaining_prob, score_label=1)
            if cascade_info["pruned_for_next_stage"]:
                print(
                    f"  Cascade stage {cascade_info['stage']}: "
                    f"目标FPR={cascade_info['target_fpr']:.4f}, "
                    f"阈值={cascade_info['threshold']:.6f}, "
                    f"下一轮多数类={cascade_info['remaining_majority']:,}"
                )
            else:
                print(f"  Cascade stage {cascade_info['stage']}: 最后一轮完成，不再淘汰样本")
        return models
    if isinstance(sampled, list):
        models = []
        for model_index, (X_sub, y_sub) in enumerate(sampled):
            rounds = _rounds_for(model_index)
            models.append(_train_one_lgb(
                X_sub, y_sub, X_valid, y_valid,
                f"{scene_label} | model {model_index+1}/{len(sampled)}",
                params_base, rounds,
            ))
        return models
    X_sampled, y_sampled = sampled
    return _train_one_lgb(
        X_sampled, y_sampled, X_valid, y_valid, scene_label, params_base, round_limits
    )


def _best_rounds(model):
    def _one(model_one):
        return int(model_one.best_iteration or model_one.current_iteration() or LGB_PARAMS["n_estimators"])
    if isinstance(model, (list, tuple)):
        return [_one(m) for m in model]
    return _one(model)


## Cell 13B：三模式配置阶段（完整比较 / 只更新轮数 / 历史复用）


In [ ]:
# ── Cell 13B：可选，只在需要重新比较采样方法时运行 ──
if RUN_MODEL_SELECTION:
    all_results = []
    trained_boosters = {}
    methods_to_run = [None] + list(dict.fromkeys(SAMPLING_METHODS))
    # ── 候选方法：只在 D_tr 采样/训练，在原始 D_val 评价 ──
    for method_order, method in enumerate(methods_to_run):
        method_key, method_label = _normalise_method_name(method)
        print(f"\n{'━'*72}\n▶ 候选场景：{method_label}\n{'━'*72}")
        candidate_params = LGB_PARAMS.copy()
        candidate_params.pop("scale_pos_weight", None)

        # 阶段1：只在60%训练集内部确定早停轮数，避免模型选择验证集参与早停。
        sampled_inner = _sample_training_data(method_key, X_inner_train, y_inner_train)
        early_stop_model = _train_sampled(
            sampled_inner, X_early_stop, y_early_stop,
            f"{method_label} | 训练集内部早停", candidate_params,
        )
        candidate_rounds = _best_rounds(early_stop_model)

        # 阶段2：按固定轮数在完整60%训练集上重新采样、重新训练候选模型。
        # 原始15%模型选择验证集只在随后评价一次，不进入训练或早停。
        sampled_train = _sample_training_data(method_key, X_train, y_train)
        candidate_model = _train_sampled(
            sampled_train, None, None,
            f"{method_label} | 完整训练集", candidate_params, candidate_rounds,
        )

        sampled_subsets = (
            sampled_train.subsets_ if isinstance(sampled_train, BalanceCascade)
            else sampled_train
        )
        if isinstance(sampled_subsets, list):
            print(f"  返回 {len(sampled_subsets)} 个训练子集")
            for subset_index, (_, y_sub) in enumerate(sampled_subsets, start=1):
                print_sampling_summary(y_train, y_sub, f"{method_label} subset{subset_index}")
            sampled_positive_rate = float(
                np.mean([np.mean(y_sub) for _, y_sub in sampled_subsets])
            )
        else:
            _, y_sampled = sampled_subsets
            if method_key != "baseline":
                print_sampling_summary(y_train, y_sampled, method_label)
            sampled_positive_rate = float(np.mean(y_sampled))
        result = evaluate_model(
            candidate_model, X_val, y_val, method_label,
            split_name="模型选择验证集",
        )
        result.update({
            "method": method_label,
            "method_key": method_key,
            "method_order": method_order,
            "train_pos_rate_after_sampling": sampled_positive_rate,
        })
        all_results.append(result)
        trained_boosters[method_label] = candidate_model

    method_selection_df = pd.DataFrame([{
        "method": result["method"],
        "method_key": result["method_key"],
        "validation_auc": result["auc"],
        "validation_precision_at_pi": result["precision"],
        "validation_recall_at_pi": result["recall"],
        "validation_f1_at_pi": result["f1"],
        "validation_threshold_at_pi": result["threshold"],
        "train_pos_rate_after_sampling": result["train_pos_rate_after_sampling"],
        "method_order": result["method_order"],
    } for result in all_results])

    top3_indices = method_selection_df.sort_values(
        ["validation_auc", "method_order"],
        ascending=[False, True], kind="mergesort",
    ).head(min(3, len(method_selection_df))).index      # 先选AUC Top 3
    method_selection_df["auc_top3"] = method_selection_df.index.isin(top3_indices)
    selected_index = method_selection_df.loc[top3_indices].sort_values(
        ["validation_precision_at_pi", "validation_auc", "method_order"],
        ascending=[False, False, True], kind="mergesort",
    ).index[0]                                          # 再从中选val上precision最高的
    method_selection_df["selected"] = method_selection_df.index == selected_index
    selected_method_key = method_selection_df.loc[selected_index, "method_key"]
    selected_method_label = method_selection_df.loc[selected_index, "method"]

    for result in all_results:
        row = method_selection_df[method_selection_df["method"] == result["method"]].iloc[0]
        result["auc_top3"] = bool(row["auc_top3"])
        result["selected"] = bool(row["selected"])

    print("\nAUC Top 3：", method_selection_df.loc[method_selection_df["auc_top3"], "method"].tolist())
    print("最终选择：", selected_method_label)
    display(method_selection_df.drop(columns=["method_order"]))
    method_selection_df.to_csv(f"sampling_method_selection_{TARGET}.csv", index=False, encoding="utf-8-sig")

    # ── 开发集 = 训练集 ∪ 模型选择验证集（约占全部样本75%），重训最优方法 ──
    X_fit = pd.concat([X_train, X_val], axis=0)
    y_fit = pd.concat([y_train, y_val], axis=0)
    development_ratio = len(X_fit) / len(df_clean)
    print(f'开发集样本数: {len(X_fit):,}，占全部样本: {development_ratio:.2%}')
    selected_candidate = trained_boosters[selected_method_label]
    selected_rounds = _best_rounds(selected_candidate)
    selected_model_config = {
        "target": TARGET,
        "selected_method_key": selected_method_key,
        "selected_method_label": selected_method_label,
        "selected_rounds": selected_rounds,
        "final_params": {k: v for k, v in LGB_PARAMS.items() if k != "scale_pos_weight"},
        "config_source": "full_method_selection",
    }
    with open(SELECTED_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(selected_model_config, f, ensure_ascii=False, indent=2)
    print(f"✅ 最优方法配置已保存：{SELECTED_CONFIG_PATH}")
    print("   selected_method_key =", repr(selected_method_key))
    print("   selected_rounds =", repr(selected_rounds))

elif REFIT_BEST_ROUNDS_ONLY:
    if FORCE_METHOD:
        selected_method_key = FORCED_METHOD_KEY
        selected_method_label = FORCED_METHOD_LABEL
        refit_params = LGB_PARAMS.copy()
        print(f"使用指定方法重新训练：{selected_method_label}")
    elif os.path.isfile(SELECTED_CONFIG_PATH):
        with open(SELECTED_CONFIG_PATH, "r", encoding="utf-8") as f:
            _previous_config = json.load(f)
        if _previous_config.get("target") != TARGET:
            raise ValueError("历史最佳配置的 TARGET 与当前 TARGET 不一致。")
        selected_method_key = _previous_config["selected_method_key"]
        selected_method_label = _previous_config["selected_method_label"]
        refit_params = (LGB_PARAMS.copy() if USE_CURRENT_LGB_PARAMS
                        else dict(_previous_config.get("final_params", LGB_PARAMS)))
        print(f"从历史配置读取已选方法：{selected_method_label}")
        if USE_CURRENT_LGB_PARAMS:
            print("使用当前单元格 LGB_PARAMS 覆盖历史参数。")
    else:
        selected_method_key = FORCED_METHOD_KEY
        selected_method_label = FORCED_METHOD_LABEL
        refit_params = LGB_PARAMS.copy()
        print(f"尚无历史配置，使用方法兜底：{selected_method_label}")
    refit_params.pop("scale_pos_weight", None)

    # 只使用60%模型训练集：其内部拟合子集采样/训练，内部早停集确定最佳轮数。
    sampled_inner = _sample_training_data(
        selected_method_key, X_inner_train, y_inner_train
    )
    rounds_model = _train_sampled(
        sampled_inner, X_early_stop, y_early_stop,
        f"仅更新最佳轮数 | {selected_method_label}", refit_params,
    )
    selected_rounds = _best_rounds(rounds_model)
    selected_model_config = {
        "target": TARGET,
        "selected_method_key": selected_method_key,
        "selected_method_label": selected_method_label,
        "selected_rounds": selected_rounds,
        "final_params": refit_params,
        "config_source": "refit_best_rounds_only",
    }
    with open(SELECTED_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(selected_model_config, f, ensure_ascii=False, indent=2)
    print(f"✅ 最佳轮数已更新并保存：{SELECTED_CONFIG_PATH}")
    print("   selected_rounds =", repr(selected_rounds))

else:
    if os.path.isfile(SELECTED_CONFIG_PATH):
        with open(SELECTED_CONFIG_PATH, "r", encoding="utf-8") as f:
            selected_model_config = json.load(f)
        if selected_model_config.get("target") != TARGET:
            raise ValueError("历史最佳配置的 TARGET 与当前 TARGET 不一致。")
        selected_method_key = selected_model_config["selected_method_key"]
        selected_method_label = selected_model_config["selected_method_label"]
        selected_rounds = selected_model_config["selected_rounds"]
        print(f"✅ 已完全复用历史配置：{SELECTED_CONFIG_PATH}")
    elif _load_saved_artifacts and os.path.isfile(METADATA_PATH):
        # 兼容旧版只保存了全量模型元数据、尚未单独保存 selected_model_config 的制品。
        with open(METADATA_PATH, "r", encoding="utf-8") as f:
            _legacy_metadata = json.load(f)
        selected_model_config = {
            "target": TARGET,
            "selected_method_key": _legacy_metadata["selected_method_key"],
            "selected_method_label": _legacy_metadata["selected_method_label"],
            "selected_rounds": _legacy_metadata["selected_rounds"],
            "final_params": _legacy_metadata.get("final_params", LGB_PARAMS),
            "config_source": "migrated_from_full_model_metadata",
        }
        selected_method_key = selected_model_config["selected_method_key"]
        selected_method_label = selected_model_config["selected_method_label"]
        selected_rounds = selected_model_config["selected_rounds"]
        with open(SELECTED_CONFIG_PATH, "w", encoding="utf-8") as f:
            json.dump(selected_model_config, f, ensure_ascii=False, indent=2)
        print(f"✅ 已从旧版全量模型元数据迁移配置：{SELECTED_CONFIG_PATH}")
    else:
        if FORCED_ROUNDS is None:
            raise FileNotFoundError(
                f"未找到 {SELECTED_CONFIG_PATH}，且 FORCED_ROUNDS=None。"
                "请先开启完整方法比较或只更新最佳轮数。"
            )
        selected_method_key = FORCED_METHOD_KEY
        selected_method_label = FORCED_METHOD_LABEL
        selected_rounds = FORCED_ROUNDS
        selected_model_config = {
            "target": TARGET,
            "selected_method_key": selected_method_key,
            "selected_method_label": selected_method_label,
            "selected_rounds": selected_rounds,
            "final_params": {k: v for k, v in LGB_PARAMS.items() if k != "scale_pos_weight"},
            "config_source": "forced_fallback",
        }
        print("⚠️ 未找到历史配置，临时使用 FORCED_* 兜底。")
    print(f"   方法：{selected_method_label}")
    print(f"   最佳轮数：{selected_rounds}")


## Cell 13C 最终模型保存和复用

保存或加载所选方法训练的模型、预处理器和元数据。


In [ ]:
# ── Cell 13C：加载制品，或按固定最优方法在全量数据上训练并保存 ──
final_params = dict(selected_model_config.get("final_params", LGB_PARAMS))
final_params.pop("scale_pos_weight", None)

if _load_saved_artifacts:
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        model_metadata = json.load(f)
    if model_metadata.get("feature_names") != list(feature_names):
        raise ValueError("保存模型的 feature_names 与当前预处理结果不一致，请检查数据/配置或改用 train。")
    selected_method_key = model_metadata["selected_method_key"]
    selected_method_label = model_metadata["selected_method_label"]
    selected_rounds = model_metadata["selected_rounds"]
    final_params = dict(model_metadata.get("final_params", final_params))
    _loaded_models = []
    for i in range(int(model_metadata["model_count"])):
        model_path = os.path.join(MODEL_DIR, f"model_{i + 1}.txt")
        if not os.path.isfile(model_path):
            raise FileNotFoundError(f"模型成员缺失：{model_path}")
        _loaded_models.append(lgb.Booster(model_file=model_path))
    full_model = _loaded_models[0] if len(_loaded_models) == 1 else _loaded_models
    print(f"✅ 已加载全量模型：{selected_method_label}，成员数={len(_loaded_models)}")
else:
    print(f"固定使用采样方法：{selected_method_label}")
    sampled_full = _sample_training_data(selected_method_key, X, y)
    full_model = _train_sampled(
        sampled_full, None, None,
        f"全量最终模型 | {selected_method_label}",
        final_params, selected_rounds,
    )
    print("✅ 全量最终模型训练完成")

    os.makedirs(MODEL_DIR, exist_ok=True)
    models = list(full_model) if isinstance(full_model, (list, tuple)) else [full_model]
    for i, model in enumerate(models):
        model.save_model(os.path.join(MODEL_DIR, f"model_{i + 1}.txt"))
    joblib.dump(model_preprocessor, PREPROCESSOR_PATH)
    model_metadata = {
        "target": TARGET,
        "selected_method_key": selected_method_key,
        "selected_method_label": selected_method_label,
        "selected_rounds": selected_rounds,
        "final_params": final_params,
        "selected_config_path": SELECTED_CONFIG_PATH,
        "feature_names": list(feature_names),
        "model_count": len(models),
        "random_state": RANDOM_STATE,
    }
    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(model_metadata, f, ensure_ascii=False, indent=2)
    print(f"✅ 已保存 {len(models)} 个模型到 {MODEL_DIR}")
    print(f"✅ 已保存预处理器：{PREPROCESSOR_PATH}")
    print(f"✅ 已保存元数据：{METADATA_PATH}")

# 兼容后续 模型 单元使用的变量名。
models = list(full_model) if isinstance(full_model, (list, tuple)) else [full_model]
final_model = full_model
booster = models[0]
trained_boosters = globals().get("trained_boosters", {})
trained_boosters["全量最终模型"] = full_model
params_run = final_params
